In [54]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import torch
import gymnasium as gym
from neuralforecast import NeuralForecast
from neuralforecast.losses.pytorch import DistributionLoss
from neuralforecast.models import TimesNet
from stable_baselines3 import A2C, PPO
from gymnasium.utils.env_checker import check_env

from config.config import appConfig
from src.data_prep import data_prep, data_for_timesnet
from src.portfolio_env import PortfolioEnv

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
config = appConfig(
    initial_balance=100000,
    allow_short_selling=False,
    technical_indicator_list=['MACD','RSI_14','CCI_20','SMA_20','EMA_20'], #'ADX_14',
    transaction_cost=0.001,
    ticker_list=["BTC-USD","ETH-USD","LTC-USD","LINK-USD","BCH-USD","UNI-USD","XLM-USD","FIL-USD","BNB-USD","SOL-USD","XRP-USD","ADA-USD","SHIB-USD","TON-USD","DOGE-USD","AVAX-USD","TRX-USD","DOT-USD","MATIC-USD","ETC-USD"],
    train_starting_date="2020-09-22",
    train_ending_date="2022-06-07",
    test_starting_date="2022-06-08",
    test_ending_date="2023-01-03",
    THRESHOLD_PARAMETER=0.015,
    oc_upper_threshold=0.01, oc_lower_threshold=-0.01, oc_k_loss=0.1, oc_k_gain=0.1, oc_n=0.725,
    ra_upper_threshold=0.01, ra_lower_threshold=-0.01, ra_k_loss=0.1, ra_k_gain=0.1, ra_n=1.22,
)


In [3]:
train_df, train_nan_dates = data_prep(config.train_starting_date, config.train_ending_date, config.ticker_list)
test_df, test_nan_dates  = data_prep(config.test_starting_date,  config.test_ending_date,  config.ticker_list)
print(f"Train shape: {train_df.shape}, dates: {train_df['ds'].nunique()}, tickers: {train_df['unique_id'].nunique()}")
print(f"Test  shape: {test_df.shape},  dates: {test_df['ds'].nunique()},  tickers: {test_df['unique_id'].nunique()}")


[*********************100%***********************]  20 of 20 completed
/mnt/c/Users/akgup/Documents/Obsidian Backup/Vault/Files/DL and NLP in Finance/Paper Implementation BBAPT/BBAPT/src/data_prep.py:37: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  data = data.stack(level = 'Ticker').reset_index()
[*********************100%***********************]  20 of 20 completed


Train shape: (7960, 14), dates: 398, tickers: 20
Test  shape: (3680, 14),  dates: 184,  tickers: 20


/mnt/c/Users/akgup/Documents/Obsidian Backup/Vault/Files/DL and NLP in Finance/Paper Implementation BBAPT/BBAPT/src/data_prep.py:37: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  data = data.stack(level = 'Ticker').reset_index()


In [48]:
model = TimesNet(h=1,
                input_size=24, #Context Window
                hidden_size = 16,
                conv_hidden_size = 32,
                loss=DistributionLoss(distribution='Normal', level=[80, 90]),
                scaler_type='standard',
                learning_rate=1e-3,
                max_steps=100,
                val_check_steps=50,
                early_stop_patience_steps=2)
nf = NeuralForecast(
    models=[model],
    freq='D'
)
nf.fit(df=data_for_timesnet(train_df), val_size=1)

Seed set to 1


GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name           | Type             | Params | Mode 
------------------------------------------------------------
0 | loss           | DistributionLoss | 5      | train
1 | padder_train   | ConstantPad1d    | 0      | train
2 | scaler         | TemporalNorm     | 0      | train
3 | model          | ModuleList       | 586 K  | train
4 | enc_embedding  | DataEmbedding    | 48     | train
5 | layer_norm     | LayerNorm        | 32     | train
6 | predict_linear | Linear           | 625    | train
7 | projection     | Linear           | 34     | train
------------------------------------------------------------
587 K     Trainable params
5         Non-trainable params
587 K     Total params
2.348     Total estimated model params size (MB)
50        Modules in train mode
0         Modules in eval mode


Sanity Checking DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]

/home/aarushgupta2735/miniconda3/envs/finrl-env/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 99: 100%|██████████| 1/1 [00:01<00:00,  0.62it/s, v_num=13, train_loss_step=0.710, train_loss_epoch=0.710, valid_loss=-0.532] 

`Trainer.fit` stopped: `max_steps=100` reached.


Epoch 99: 100%|██████████| 1/1 [00:01<00:00,  0.61it/s, v_num=13, train_loss_step=0.710, train_loss_epoch=0.710, valid_loss=-0.532]


Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
/home/aarushgupta2735/miniconda3/envs/finrl-env/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00,  3.02it/s]


In [57]:
test_df

Attributes,ds,unique_id,Close,High,Low,Open,Volume,y,MACD,RSI_14,CCI_20,SMA_20,EMA_20,std_dev
500,2022-07-03,ADA-USD,0.455587,0.458882,0.444546,0.456321,439469536,-0.186027,-0.030547,35.278026,-109.075850,0.477932,0.488829,0.050881
501,2022-07-03,AVAX-USD,16.660402,16.754839,15.868222,16.336060,242175995,-7.839851,-1.119607,40.357708,-49.282402,17.427042,18.179486,2.599899
502,2022-07-03,BCH-USD,105.580055,107.246307,102.156685,106.455315,1187978767,-71.458931,-13.216929,27.892495,-99.045477,113.316893,117.607476,22.305221
503,2022-07-03,BNB-USD,218.980316,219.788712,214.464493,218.047577,724525676,-69.635773,-10.934925,38.120491,-30.054043,221.982262,229.973579,25.060204
504,2022-07-03,BTC-USD,19297.076172,19558.269531,18966.951172,19242.095703,16390821947,-10917.279297,-1886.997964,24.285834,-114.793304,20503.078613,21379.754949,3509.309707
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4175,2023-01-02,TON-USD,1.060018,1.064375,1.051700,1.058733,646689,-1.460720,-0.037108,37.578046,-61.949949,1.092349,1.095317,0.344643
4176,2023-01-02,TRX-USD,0.055157,0.055394,0.054505,0.054818,138920680,-0.025480,0.000100,54.710841,66.612380,0.054576,0.054482,0.006129
4177,2023-01-02,UNI-USD,0.000100,0.000101,0.000099,0.000100,5,-0.000051,-0.000433,45.377982,-56.621005,0.000101,0.000213,0.056472
4178,2023-01-02,XLM-USD,0.073718,0.074293,0.071320,0.072519,59178367,-0.066438,-0.003314,39.604475,-63.480905,0.074924,0.075601,0.015261


In [55]:
gym.register(
    id='PortfolioEnv-v4',
    entry_point=PortfolioEnv,
    kwargs={'data': train_df, 'config': config}
)
env = gym.make('PortfolioEnv-v4')
try:
    check_env(env.unwrapped)
    print("Environment passes all checks!")
except Exception as e:
    print(f"Environment has issues: {e}")


Environment passes all checks!


/home/aarushgupta2735/miniconda3/envs/finrl-env/lib/python3.10/site-packages/gymnasium/envs/registration.py:637: UserWarning: WARN: Overriding environment PortfolioEnv-v4 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
/home/aarushgupta2735/miniconda3/envs/finrl-env/lib/python3.10/site-packages/gymnasium/utils/env_checker.py:317: UserWarning: WARN: A Box observation space minimum value is -infinity. This is probably too low.
  logger.warn(
/home/aarushgupta2735/miniconda3/envs/finrl-env/lib/python3.10/site-packages/gymnasium/utils/env_checker.py:321: UserWarning: WARN: A Box observation space maximum value is infinity. This is probably too high.
  logger.warn(


In [ ]:
model_rl = A2C('MlpPolicy', env, verbose=1)
model_rl.learn(total_timesteps=10000)
print("Training completed!")

Box(-inf, inf, (20, 7), float32)

In [ ]:
list_forecasts = []
for ticker in config.ticker_list:
    nf.fit(df=data_for_timesnet(train_df.loc[train_df['unique_id'] == ticker]), val_size=1)
    list_forecasts.append(nf.predict())
forecast = pd.concat(list_forecasts,ignore_index=True)
forecast['avg_return'] = forecast.groupby('ds')['TimesNet'].transform('mean')